In [1]:
!pip install git+https://github.com/huggingface/transformers torch accelerate bitsandbytes langchain
!pip install evaluate
!pip install datasets
!pip install -U sentence-transformers
!pip install langchain_community
!pip install peft
!pip install rouge_score
!pip install bert_score
!pip install faiss-gpu

  Cloning https://github.com/huggingface/transformers to /tmp/pip-req-build-cmsast9q
  Running command git clone --filter=blob:none --quiet https://github.com/huggingface/transformers /tmp/pip-req-build-cmsast9q
  Resolved https://github.com/huggingface/transformers to commit 52cb4034ada381fe1ffe8d428a1076e5411a8026
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.4/50.4 kB 762.6 kB/s eta 0:00:00 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 MB 11.2 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 997.8/997.8 kB 42.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 391.5/391.5 kB 19.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.4/140.4 kB 8.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.0/54.0 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 141

In [2]:
from langchain.vectorstores import FAISS

In [3]:
!pip install langchain-experimental
from langchain_experimental.text_splitter import SemanticChunker
from langchain.embeddings.sentence_transformer import SentenceTransformerEmbeddings

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 204.3/204.3 kB 1.5 MB/s eta 0:00:00a 0:00:01


In [5]:
import pandas as pd
df = pd.read_csv("/kaggle/input/coovid/covid_abstracts.csv")
df = df["abstract"]

In [6]:
from langchain_experimental.text_splitter import SemanticChunker
from langchain.embeddings.sentence_transformer import SentenceTransformerEmbeddings

In [7]:
from langchain.embeddings.sentence_transformer import SentenceTransformerEmbeddings
embedding_model = SentenceTransformerEmbeddings(model_name='BAAI/bge-large-zh-v1.5')

/opt/conda/lib/python3.10/site-packages/langchain_core/_api/deprecation.py:141: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 0.3.0. An updated version of the class exists in the langchain-huggingface package and should be used instead. To use it run `pip install -U langchain-huggingface` and import as `from langchain_huggingface import HuggingFaceEmbeddings`.
  warn_deprecated(
/opt/conda/lib/python3.10/site-packages/sentence_transformers/cross_encoder/CrossEncoder.py:11: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm, trange
2024-08-19 04:14:58.499310: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2024-08-19 04:14:58.499446: E external/local_x

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/30.4k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/394 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/110k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/439k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

1_Pooling/config.json:   0%|          | 0.00/191 [00:00<?, ?B/s]

In [8]:
text_splitter = SemanticChunker(
    embedding_model, breakpoint_threshold_type="percentile"
)
from langchain.docstore.document import Document
docs = [Document(page_content=entry) for entry in df]
docs = text_splitter.split_documents(docs)

In [9]:
len(df)

10000

In [10]:
faiss_db = FAISS.from_documents(docs, embedding_model)

In [11]:
import torch
from transformers import pipeline, AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.bfloat16
)

In [12]:
from transformers import pipeline, AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import torch
from langchain.llms import HuggingFacePipeline
from langchain.chains import LLMChain
from langchain.prompts import PromptTemplate
device = 'cuda' if torch.cuda.is_available() else 'cpu'

In [13]:
from langchain.docstore.document import Document
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.vectorstores import FAISS
from langchain.chains import RetrievalQA
from langchain.embeddings.sentence_transformer import SentenceTransformerEmbeddings
from peft import PeftModel
from langchain.retrievers import EnsembleRetriever

In [14]:
model_name = "filipealmeida/Mistral-7B-Instruct-v0.1-sharded"
base_model = AutoModelForCausalLM.from_pretrained(model_name,
                                             torch_dtype=torch.bfloat16,
                                             quantization_config=bnb_config)
tokenizer = AutoTokenizer.from_pretrained(model_name)

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

`low_cpu_mem_usage` was None, now set to True since model is quantized.


pytorch_model.bin.index.json:   0%|          | 0.00/23.9k [00:00<?, ?B/s]

pytorch_model-00001-of-00008.bin:   0%|          | 0.00/1.89G [00:00<?, ?B/s]

pytorch_model-00002-of-00008.bin:   0%|          | 0.00/1.95G [00:00<?, ?B/s]

...

generation_config.json:   0%|          | 0.00/111 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.47k [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/493k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.80M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/72.0 [00:00<?, ?B/s]

In [15]:
!pip install rank_bm25

/opt/conda/lib/python3.10/pty.py:89: RuntimeWarning: os.fork() was called. os.fork() is incompatible with multithreaded code, and JAX is multithreaded, so this will likely lead to a deadlock.
  pid, fd = os.forkpty()
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


In [16]:
from langchain.schema import Document
from langchain_community.retrievers import BM25Retriever
from typing import List

def custom_preprocessing_func(text: str) -> List[str]: # Change the input type to str
    return text.split()

bm25_retriever = BM25Retriever.from_texts(
    [doc.page_content for doc in docs],
    preprocess_func=custom_preprocessing_func
)
bm25_retriever.k = 2

In [17]:
faiss_retriever = faiss_db.as_retriever(
    search_kwargs={'k': 2}
)

In [18]:
peft_model_id = "ProElectro07/Projectxx2"
model = PeftModel.from_pretrained(base_model, peft_model_id)

adapter_config.json:   0%|          | 0.00/719 [00:00<?, ?B/s]

adapter_model.safetensors:   0%|          | 0.00/431M [00:00<?, ?B/s]

In [19]:
tokenizer.pad_token = tokenizer.eos_token

In [20]:
from langchain.chains import SimpleSequentialChain, LLMChain

In [21]:
##############
text_generation_pipeline = pipeline(
    task="text-generation",
    model=base_model,
    tokenizer=tokenizer,
    temperature=0.3,
    top_k=10,
    top_p=.85,
    repetition_penalty=1.1,
    return_full_text=False,
    max_new_tokens=500,
    num_return_sequences=1,
    # truncation=False,
    do_sample=True,
    # no_repeat_ngram_size=3,
    # early_stopping=True
)

text_generation_pipeline.model = model

mistral_llm = HuggingFacePipeline(pipeline=text_generation_pipeline)

/opt/conda/lib/python3.10/site-packages/langchain_core/_api/deprecation.py:141: LangChainDeprecationWarning: The class `HuggingFacePipeline` was deprecated in LangChain 0.0.37 and will be removed in 0.3. An updated version of the class exists in the langchain-huggingface package and should be used instead. To use it run `pip install -U langchain-huggingface` and import as `from langchain_huggingface import HuggingFacePipeline`.
  warn_deprecated(


In [22]:

from langchain.retrievers.multi_query import MultiQueryRetriever

ensemble_retriever = EnsembleRetriever(
    retrievers=[bm25_retriever, faiss_retriever],
    weights=[.3, 0.7]
)

# retriever_from_llm = MultiQueryRetriever.from_llm(
#     retriever=ensemble_retriever, llm=mistral_llm
# )

In [23]:
from bert_score import score
from evaluate import load
rouge = load('rouge')
bertscore = load('bertscore')
bleu = load('bleu')

In [24]:
ds = pd.read_csv("/kaggle/input/desfcriptionn/validate_dataset.csv")

In [25]:
ds = ds["description"]

In [26]:
predictions = []

In [27]:
from langchain import PromptTemplate, LLMChain
from langchain.chains import SimpleSequentialChain
from langchain.chains import RetrievalQA

In [29]:
torch.backends.cuda.enable_mem_efficient_sdp(False)
torch.backends.cuda.enable_flash_sdp(False)


In [30]:
prompt_template = ("""[INST]
You are an expert in evaluating the relevance of documents to a query.

Given the query and the 4 documents below:

Query: {question}
Documents: {context}

Task:
1. For each document, label it as "Relevant" if it helps to answer the query, or "Irrelevant" if it does not.

Example:
Relevant
Relevant
Irrelevant
Irrelevant

Please provide the labels only, with each of the 4 document's label on a new line like the above format.

[/INST]
""")



# Create a prompt instance
Prompt_Template = PromptTemplate.from_template(prompt_template)

# Create the RAG chain
RAG_chain = RetrievalQA.from_chain_type(
    mistral_llm,
    retriever=ensemble_retriever,
    chain_type_kwargs={"prompt": Prompt_Template}
)

predictions = []

for query in ds:
    response = RAG_chain({"query":query})
    print(response["result"])
    predictions.append(response["result"])

Relevant
Relevant
Irrelevant
Irrelevant
Relevant
Irrelevant
Irrelevant
Relevant
Relevant
Irrelevant
Irrelevant
Irrelevant
Relevant
Irrelevant
Irrelevant
Irrelevant
Relevant
Irrelevant
Irrelevant
Irrelevant
Relevant
Irrelevant
Irrelevant
Irrelevant
Relevant
Irrelevant
Irrelevant
Irrelevant
Relevant
Irrelevant
Irrelevant
Relevant
Relevant
Relevant
Irrelevant
Irrelevant


You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


Relevant
Irrelevant
Irrelevant
Irrelevant
Relevant
Irrelevant
Irrelevant
Irrelevant
Relevant
Relevant
Irrelevant
Irrelevant
Relevant
Irrelevant
Irrelevant
Relevant
Relevant
Irrelevant
Irrelevant
Relevant
Relevant
Irrelevant
Irrelevant
Relevant
Relevant
Irrelevant
Irrelevant
Irrelevant
Relevant
Irrelevant
Irrelevant
Irrelevant
Relevant
Irrelevant
Irrelevant
Irrelevant
Relevant
Relevant
Irrelevant
Irrelevant
Relevant
Relevant
Irrelevant
Irrelevant
Relevant
Relevant
Irrelevant
Irrelevant
Relevant
Relevant
Irrelevant
Irrelevant
Relevant
Relevant
Irrelevant
Irrelevant
Relevant
Relevant
Irrelevant
Irrelevant
Relevant
Irrelevant
Irrelevant
Relevant
Relevant
Irrelevant
Irrelevant
Irrelevant
Relevant
Relevant
Irrelevant
Irrelevant
Relevant
Irrelevant
Irrelevant
Irrelevant
Relevant
Irrelevant
Irrelevant
Irrelevant
Relevant
Irrelevant
Irrelevant
Irrelevant
Relevant
Relevant
Irrelevant
Irrelevant
Relevant
Relevant
Irrelevant
Irrelevant
Relevant
Irrelevant
Irrelevant
Relevant
Relevant
Irrelevant
Ir

In [31]:
predictions

['Relevant\nRelevant\nIrrelevant\nIrrelevant',
 'Relevant\nIrrelevant\nIrrelevant\nRelevant',
 'Relevant\nIrrelevant\nIrrelevant\nIrrelevant',
 'Relevant\nIrrelevant\nIrrelevant\nIrrelevant',
 'Relevant\nIrrelevant\nIrrelevant\nIrrelevant',
 'Relevant\nIrrelevant\nIrrelevant\nIrrelevant',
 'Relevant\nIrrelevant\nIrrelevant\nIrrelevant',
 'Relevant\nIrrelevant\nIrrelevant\nRelevant',
 'Relevant\nRelevant\nIrrelevant\nIrrelevant',
 'Relevant\nIrrelevant\nIrrelevant\nIrrelevant',
 'Relevant\nIrrelevant\nIrrelevant\nIrrelevant',
 'Relevant\nRelevant\nIrrelevant\nIrrelevant',
 'Relevant\nIrrelevant\nIrrelevant\nRelevant',
 'Relevant\nIrrelevant\nIrrelevant\nRelevant',
 'Relevant\nIrrelevant\nIrrelevant\nRelevant',
 'Relevant\nIrrelevant\nIrrelevant\nIrrelevant',
 'Relevant\nIrrelevant\nIrrelevant\nIrrelevant',
 'Relevant\nIrrelevant\nIrrelevant\nIrrelevant',
 'Relevant\nRelevant\nIrrelevant\nIrrelevant',
 'Relevant\nRelevant\nIrrelevant\nIrrelevant',
 'Relevant\nRelevant\nIrrelevant\nIrrele

In [32]:
p = predictions

In [33]:
responses = p

d = []

for response in responses:
  labels = response.strip().lower().split('\n')
  d.append(labels)

In [34]:
d

[['relevant', 'relevant', 'irrelevant', 'irrelevant'],
 ['relevant', 'irrelevant', 'irrelevant', 'relevant'],
 ['relevant', 'irrelevant', 'irrelevant', 'irrelevant'],
 ['relevant', 'irrelevant', 'irrelevant', 'irrelevant'],
 ['relevant', 'irrelevant', 'irrelevant', 'irrelevant'],
 ['relevant', 'irrelevant', 'irrelevant', 'irrelevant'],
 ['relevant', 'irrelevant', 'irrelevant', 'irrelevant'],
 ['relevant', 'irrelevant', 'irrelevant', 'relevant'],
 ['relevant', 'relevant', 'irrelevant', 'irrelevant'],
 ['relevant', 'irrelevant', 'irrelevant', 'irrelevant'],
 ['relevant', 'irrelevant', 'irrelevant', 'irrelevant'],
 ['relevant', 'relevant', 'irrelevant', 'irrelevant'],
 ['relevant', 'irrelevant', 'irrelevant', 'relevant'],
 ['relevant', 'irrelevant', 'irrelevant', 'relevant'],
 ['relevant', 'irrelevant', 'irrelevant', 'relevant'],
 ['relevant', 'irrelevant', 'irrelevant', 'irrelevant'],
 ['relevant', 'irrelevant', 'irrelevant', 'irrelevant'],
 ['relevant', 'irrelevant', 'irrelevant', 'irre

In [35]:
for k in d:
  print(len(k))

4
4
4
4
4
4
4
4
4
4
4
4
4
4
4
4
4
4
4
4
4
4
4
4
4
4
4
4
4
4
4
4
4
4
4
4
4
4
4
4
4
4
4
4
4
4
4
4
4
4


In [36]:
score = []

test = 0

for i in d:
  t = 0
  if i[0]=="relevant":
    t = t + .4
  if i[1]=="relevant":
    t = t + .3
  if i[2]=="relevant":
    t = t + .2
  if i[3]=="relevant":
    t = t + .1
  score.append(t)
  test = test + t

In [37]:
(score)

[0.7,
 0.5,
 0.4,
 0.4,
 0.4,
 0.4,
 0.4,
 0.5,
 0.7,
 0.4,
 0.4,
 0.7,
 0.5,
 0.5,
 0.5,
 0.4,
 0.4,
 0.4,
 0.7,
 0.7,
 0.7,
 0.7,
 0.7,
 0.7,
 0.5,
 0.4,
 0.7,
 0.4,
 0.4,
 0.4,
 0.7,
 0.7,
 0.5,
 0.5,
 0.5,
 0.4,
 0.5,
 0.7,
 0.5,
 0.4,
 0.4,
 0.6000000000000001,
 0.5,
 0.7,
 0.5,
 0.6000000000000001,
 0.7,
 0.4,
 0.7,
 0.4]

In [38]:
d[-5]

['relevant', 'irrelevant', 'relevant', 'irrelevant']

In [39]:
(test/50)*100

52.99999999999998

In [40]:
prompt_template = ("""[INST]
You are an expert in evaluating the relevance of documents to a query.

Given the query and the 4 documents below:

Query: {question}
Documents: {context}

Task:
1. For each document, label it as "Relevant" if it helps to answer the query, or "Irrelevant" if it does not.

Example:
Relevant
Relevant
Irrelevant
Irrelevant

Please provide the labels only, with each of the 4 document's label on a new line like the above format.

[/INST]
""")



# Create a prompt instance
Prompt_Template = PromptTemplate.from_template(prompt_template)

# Create the RAG chain
RAG_chain = RetrievalQA.from_chain_type(
    mistral_llm,
    retriever=faiss_retriever,
    chain_type_kwargs={"prompt": Prompt_Template}
)

predictions = []

for query in ds:
    response = RAG_chain({"query":query})
    print(response["result"])
    predictions.append(response["result"])

Relevant
Relevant
Irrelevant
Irrelevant
Relevant
Relevant
Irrelevant
Irrelevant
Relevant
Relevant
Irrelevant
Irrelevant
Relevant
Irrelevant
Irrelevant
Irrelevant
Relevant
Irrelevant
Irrelevant
Relevant
Relevant
Relevant
Irrelevant
Irrelevant
Relevant
Irrelevant
Irrelevant
Relevant
Relevant
Irrelevant
Irrelevant
Irrelevant
Relevant
Relevant
Irrelevant
Irrelevant
Relevant
Irrelevant
Irrelevant
Irrelevant
Relevant
Relevant
Irrelevant
Irrelevant
Relevant
Irrelevant
Irrelevant
Relevant
Relevant
Irrelevant
Irrelevant
Relevant
Relevant
Irrelevant
Irrelevant
Relevant
Relevant
Irrelevant
Irrelevant
Relevant
Relevant
Irrelevant
Irrelevant
Relevant
Relevant
Irrelevant
Irrelevant
Irrelevant
Relevant
Relevant
Irrelevant
Irrelevant
Relevant
Irrelevant
Irrelevant
Relevant
Relevant
Relevant
Irrelevant
Irrelevant
Relevant
Irrelevant
Irrelevant
Relevant
Relevant
Relevant
Irrelevant
Irrelevant
Relevant
Relevant
Irrelevant
Irrelevant
Relevant
Relevant
Irrelevant
Irrelevant
Relevant
Irrelevant
Irrelevant
I

In [41]:
predictions

['Relevant\nRelevant\nIrrelevant\nIrrelevant',
 'Relevant\nRelevant\nIrrelevant\nIrrelevant',
 'Relevant\nRelevant\nIrrelevant\nIrrelevant',
 'Relevant\nIrrelevant\nIrrelevant\nIrrelevant',
 'Relevant\nIrrelevant\nIrrelevant\nRelevant',
 'Relevant\nRelevant\nIrrelevant\nIrrelevant',
 'Relevant\nIrrelevant\nIrrelevant\nRelevant',
 'Relevant\nIrrelevant\nIrrelevant\nIrrelevant',
 'Relevant\nRelevant\nIrrelevant\nIrrelevant',
 'Relevant\nIrrelevant\nIrrelevant\nIrrelevant',
 'Relevant\nRelevant\nIrrelevant\nIrrelevant',
 'Relevant\nIrrelevant\nIrrelevant\nRelevant',
 'Relevant\nIrrelevant\nIrrelevant\nRelevant',
 'Relevant\nIrrelevant\nIrrelevant\nRelevant',
 'Relevant\nIrrelevant\nIrrelevant\nRelevant',
 'Relevant\nIrrelevant\nIrrelevant\nRelevant',
 'Relevant\nIrrelevant\nIrrelevant\nIrrelevant',
 'Relevant\nRelevant\nIrrelevant\nIrrelevant',
 'Relevant\nIrrelevant\nIrrelevant\nRelevant',
 'Relevant\nRelevant\nIrrelevant\nIrrelevant',
 'Relevant\nIrrelevant\nIrrelevant\nRelevant',
 'Rel

In [42]:
p = predictions

In [43]:
responses = p

d = []

for response in responses:
  labels = response.strip().lower().split('\n')
  d.append(labels)

In [44]:
for k in d:
  print(len(k))

4
4
4
4
4
4
4
4
4
4
4
4
4
4
4
4
4
4
4
4
4
4
4
4
4
4
4
4
4
4
4
4
4
4
4
4
4
4
4
4
4
4
4
4
4
4
4
4
4
4


In [45]:
score = []

test = 0

for i in d:
  t = 0
  if i[0]=="relevant":
    t = t + .4
  if i[1]=="relevant":
    t = t + .3
  if i[2]=="relevant":
    t = t + .2
  if i[3]=="relevant":
    t = t + .1
  score.append(t)
  test = test + t

In [46]:
score

[0.7,
 0.7,
 0.7,
 0.4,
 0.5,
 0.7,
 0.5,
 0.4,
 0.7,
 0.4,
 0.7,
 0.5,
 0.5,
 0.5,
 0.5,
 0.5,
 0.4,
 0.7,
 0.5,
 0.7,
 0.5,
 0.7,
 0.7,
 0.7,
 0.4,
 0.7,
 0.5,
 0.5,
 0.5,
 0.7,
 0.7,
 0.7,
 0.6000000000000001,
 0.6000000000000001,
 0.6000000000000001,
 0.4,
 0.4,
 0.5,
 0.4,
 0.5,
 0.5,
 0.4,
 0.7,
 0.4,
 0.7,
 0.4,
 0.7,
 0.4,
 0.4,
 0.5]

In [47]:
(test/50)*100

55.199999999999974

In [48]:
prompt_template = ("""[INST]
You are an expert in evaluating the relevance of documents to a query.

Given the query and the 4 documents below:

Query: {question}
Documents: {context}

Task:
1. For each document, label it as "Relevant" if it helps to answer the query, or "Irrelevant" if it does not.

Example:
Relevant
Relevant
Irrelevant
Irrelevant

Please provide the labels only, with each of the 4 document's label on a new line like the above format.

[/INST]
""")



# Create a prompt instance
Prompt_Template = PromptTemplate.from_template(prompt_template)

# Create the RAG chain
RAG_chain = RetrievalQA.from_chain_type(
    mistral_llm,
    retriever=bm25_retriever,
    chain_type_kwargs={"prompt": Prompt_Template}
)

predictions = []

for query in ds:
    response = RAG_chain({"query":query})
    print(response["result"])
    predictions.append(response["result"])

Relevant
Irrelevant
Irrelevant
Relevant
Relevant
Irrelevant
Irrelevant
Relevant
Relevant
Irrelevant
Irrelevant
Irrelevant
Relevant
Irrelevant
Irrelevant
Irrelevant
Relevant
Irrelevant
Irrelevant
Relevant
Relevant
Irrelevant
Irrelevant
Relevant
Relevant
Irrelevant
Irrelevant
Irrelevant
Relevant
Irrelevant
Irrelevant
Relevant
Relevant
Irrelevant
Irrelevant
Relevant
Relevant
Irrelevant
Irrelevant
Relevant
Relevant
Relevant
Irrelevant
Irrelevant
Relevant
Relevant
Irrelevant
Irrelevant
Relevant
Irrelevant
Irrelevant
Relevant
Relevant
Irrelevant
Irrelevant
Relevant
Relevant
Irrelevant
Irrelevant
Relevant
Relevant
Irrelevant
Irrelevant
Relevant
Relevant
Relevant
Irrelevant
Irrelevant
Relevant
Relevant
Irrelevant
Irrelevant
Relevant
Irrelevant
Irrelevant
Relevant
Relevant
Irrelevant
Irrelevant
Relevant
Relevant
Relevant
Irrelevant
Irrelevant
Relevant
Relevant
Irrelevant
Irrelevant
Relevant
Irrelevant
Irrelevant
Irrelevant
Relevant
Irrelevant
Irrelevant
Relevant
Relevant
Irrelevant
Irrelevant
R

In [49]:
p = predictions

In [50]:
responses = p

d = []

for response in responses:
  labels = response.strip().lower().split('\n')
  d.append(labels)

In [51]:
score = []

test = 0

for i in d:
  t = 0
  if i[0]=="relevant":
    t = t + .4
  if i[1]=="relevant":
    t = t + .3
  if i[2]=="relevant":
    t = t + .2
  if i[3]=="relevant":
    t = t + .1
  score.append(t)
  test = test + t

In [52]:
(test/50)*100

52.39999999999998

In [53]:
d

[['relevant', 'irrelevant', 'irrelevant', 'relevant'],
 ['relevant', 'irrelevant', 'irrelevant', 'relevant'],
 ['relevant', 'irrelevant', 'irrelevant', 'irrelevant'],
 ['relevant', 'irrelevant', 'irrelevant', 'irrelevant'],
 ['relevant', 'irrelevant', 'irrelevant', 'relevant'],
 ['relevant', 'irrelevant', 'irrelevant', 'relevant'],
 ['relevant', 'irrelevant', 'irrelevant', 'irrelevant'],
 ['relevant', 'irrelevant', 'irrelevant', 'relevant'],
 ['relevant', 'irrelevant', 'irrelevant', 'relevant'],
 ['relevant', 'irrelevant', 'irrelevant', 'relevant'],
 ['relevant', 'relevant', 'irrelevant', 'irrelevant'],
 ['relevant', 'relevant', 'irrelevant', 'irrelevant'],
 ['relevant', 'irrelevant', 'irrelevant', 'relevant'],
 ['relevant', 'irrelevant', 'irrelevant', 'relevant'],
 ['relevant', 'irrelevant', 'irrelevant', 'relevant'],
 ['relevant', 'irrelevant', 'irrelevant', 'relevant'],
 ['relevant', 'relevant', 'irrelevant', 'irrelevant'],
 ['relevant', 'relevant', 'irrelevant', 'irrelevant'],
 ['r